# Lecture 4: Weight initialization

<div class="alert alert-success">This interactive notebook demonstrates the effect of weight initialization on activations and gradients in deep networks. Use the controls to explore how different initialization schemes, network depths, and activation functions affect signal propagation.</div>

*Inspired by the [https://www.deeplearning.ai/ai-notes/initialization/](https://www.deeplearning.ai/ai-notes/initialization/).*

In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## Interactive exploration

The network is a fully-connected MLP with a configurable number of layers, all of width 256. A batch of random Gaussian inputs is passed through the network, and we record the activations (after the nonlinearity) and the gradients (of the output w.r.t. each layer's pre-activation) at every layer.

In [2]:
def build_and_probe(n_layers, width, init_scheme, activation, batch_size=512, input_dim=256):

    # Build layers
    act_fn = {'ReLU': nn.ReLU, 'Tanh': nn.Tanh, 'Sigmoid': nn.Sigmoid}[activation]
    layers = []
    dims = [input_dim] + [width] * n_layers
    for i in range(n_layers):
        layers.append(nn.Linear(dims[i], dims[i + 1]))
        layers.append(act_fn())
    model = nn.Sequential(*layers)

    # Initialize
    for m in model:
        if isinstance(m, nn.Linear):
            if init_scheme == 'Zero':
                nn.init.zeros_(m.weight)
            elif init_scheme == 'Uniform(-1, 1)':
                nn.init.uniform_(m.weight, -1, 1)
            elif init_scheme == 'N(0, 0.01)':
                nn.init.normal_(m.weight, 0, 0.01)
            elif init_scheme == 'N(0, 1)':
                nn.init.normal_(m.weight, 0, 1.0)
            elif init_scheme == 'Xavier':
                nn.init.xavier_normal_(m.weight)
            elif init_scheme == 'He (Kaiming)':
                nonlin = 'relu' if activation == 'ReLU' else 'tanh'
                try:
                    nn.init.kaiming_normal_(m.weight, nonlinearity=nonlin)
                except ValueError:
                    nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            nn.init.zeros_(m.bias)

    # Forward pass, recording activations
    x = torch.randn(batch_size, input_dim)
    activations = []
    pre_activations = []
    h = x
    for layer in model:
        h = layer(h)
        if isinstance(layer, (nn.ReLU, nn.Tanh, nn.Sigmoid)):
            activations.append(h.detach().numpy().flatten())

    # Backward pass for gradients
    x = torch.randn(batch_size, input_dim)
    x.requires_grad_(False)
    pre_acts = []
    h = x
    for layer in model:
        h = layer(h)
        if isinstance(layer, nn.Linear):
            h.retain_grad()
            pre_acts.append(h)

    loss = h.sum()
    loss.backward()

    gradients = []
    for pa in pre_acts:
        if pa.grad is not None:
            gradients.append(pa.grad.detach().numpy().flatten())
        else:
            gradients.append(np.array([0.0]))

    return activations, gradients

In [3]:
# Widgets
w_layers = widgets.IntSlider(value=10, min=2, max=30, step=1, description='Layers:')
w_init = widgets.Dropdown(
    options=['Zero', 'Uniform(-1, 1)', 'N(0, 0.01)', 'N(0, 1)', 'Xavier', 'He (Kaiming)'],
    value='He (Kaiming)', description='Init:'
)
w_act = widgets.Dropdown(
    options=['ReLU', 'Tanh', 'Sigmoid'],
    value='ReLU', description='Activation:'
)
w_show = widgets.ToggleButtons(
    options=['Activations', 'Gradients'],
    value='Activations', description='Show:'
)

output = widgets.Output()

def update(change=None):
    n_layers = w_layers.value
    init_scheme = w_init.value
    activation = w_act.value
    show = w_show.value

    activations, gradients = build_and_probe(n_layers, 256, init_scheme, activation)
    data = activations if show == 'Activations' else gradients

    # Pick layers to display (evenly spaced, max 8)
    n = len(data)
    if n <= 8:
        indices = list(range(n))
    else:
        indices = [int(i) for i in np.linspace(0, n - 1, 8)]

    with output:
        output.clear_output(wait=True)
        fig, axes = plt.subplots(1, len(indices), figsize=(2.5 * len(indices), 3))
        if len(indices) == 1:
            axes = [axes]

        for ax, idx in zip(axes, indices):
            d = data[idx]
            d = d[np.isfinite(d)]
            if len(d) == 0 or np.std(d) == 0:
                ax.text(0.5, 0.5, 'dead\n(all zeros)' if np.all(d == 0) else 'exploded',
                        transform=ax.transAxes, ha='center', va='center',
                        fontsize=9, color='red')
                ax.set_xlim(-1, 1)
            else:
                ax.hist(d, bins=50, density=True, alpha=0.7, color='steelblue')
            ax.set_title(f'Layer {idx + 1}', fontsize=10)
            ax.set_ylim(bottom=0)
            ax.tick_params(labelsize=7)
            if idx != indices[0]:
                ax.set_yticks([])

        fig.suptitle(f'{show} — {init_scheme} init, {activation}, {n_layers} layers',
                     fontsize=12, y=1.02)
        plt.tight_layout()
        plt.show()

        # Summary stats
        means = [np.mean(np.abs(d[np.isfinite(d)])) if len(d[np.isfinite(d)]) > 0 else 0 for d in data]
        stds = [np.std(d[np.isfinite(d)]) if len(d[np.isfinite(d)]) > 0 else 0 for d in data]

        fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
        ax1.bar(range(1, len(means) + 1), means, color='steelblue', alpha=0.7)
        ax1.set_xlabel('Layer')
        ax1.set_ylabel('Mean |value|')
        ax1.set_title(f'Mean |{show.lower()[:-1]}| per layer')

        ax2.bar(range(1, len(stds) + 1), stds, color='coral', alpha=0.7)
        ax2.set_xlabel('Layer')
        ax2.set_ylabel('Std')
        ax2.set_title(f'Std of {show.lower()} per layer')

        plt.tight_layout()
        plt.show()

w_layers.observe(update, names='value')
w_init.observe(update, names='value')
w_act.observe(update, names='value')
w_show.observe(update, names='value')

display(widgets.HBox([w_layers, w_init, w_act, w_show]))
display(output)
update()

Output()